# 方案三：CI-NEB 动力学反应路径


## 1. 参数配置

In [ ]:
# ===== 修复 OpenMP 冲突 =====
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ===== 目标化合物 =====
TARGET_FORMULA = "Li2O"

# ===== 中间相热力学过滤（仅筛除实验不可及相）=====
E_ABOVE_HULL_CUTOFF = 0.2   # eV/atom, 三元体系建议0.15-0.2，二元0.1
COMPOSITION_WINDOW = True   # True=组分窗口过滤, False=不限制

# ===== 手动指定前驱体 =====
USE_MANUAL_PRECURSORS = False   # True=启用, False=凸包自动
MANUAL_PRECURSORS = ["Li", "O2"]  # 手动指定前驱体列表

# ===== debug mode =====
DEBUG_MODE = False    # True=只跑前N条边快速调试, False=完整运行
DEBUG_COARSE_EDGES = 5     # 调试模式下最多跑多少条粗筛边
DEBUG_FINE_EDGES = 3       # 调试模式下最多精修多少条边
MAX_REACTANTS = 2            # 最大同时反应相数 n=2
K_SHORTEST = 10              # KSP 路径数

# ===== 粗筛 CI-NEB 参数（全边）=====
COARSE_N_IMAGES = 5
COARSE_NEB_FMAX = 1.0
COARSE_NEB_STEPS = 200

# ===== 精修 CI-NEB 参数（仅Top-3路径边）=====
FINE_N_IMAGES = 7
FINE_NEB_FMAX = 0.5
FINE_NEB_STEPS = 500

# ===== MACE 设置 =====
MACE_MODEL = "medium"
MACE_DTYPE = "float64"

# ===== GPU 加速 =====
USE_GPU = True           # True=启用GPU加速, False=只用CPU
MACE_DEVICE = "cpu"      # GPU检测逻辑移至Cell 11



## 2. 导入库

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
from mp_api.client import MPRester
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import networkx as nx
from itertools import combinations
from math import gcd
from collections import defaultdict
from time import time

import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

from pymatgen.analysis.phase_diagram import PhaseDiagram
from pymatgen.entries.mixing_scheme import MaterialsProjectDFTMixingScheme
from pymatgen.io.ase import AseAtomsAdaptor
from ase.mep import idpp_interpolate
from ase.mep.neb import NEB
from ase.optimize import FIRE
from mace.calculators.foundations_models import mace_mp
from rxn_network.reactions.computed import ComputedReaction
from scipy.optimize import linear_sum_assignment

load_dotenv(os.path.join("..", "api", "myapi.env"))
API_KEY = os.getenv("MP_API_KEY")
if not API_KEY: raise ValueError("MP_API_KEY missing")
print("All imports OK, API key loaded")

# --- Parameter summary ---
print(f"Target: {TARGET_FORMULA} | Hull cutoff: {E_ABOVE_HULL_CUTOFF} eV/atom")
print(f"Coarse NEB: {COARSE_N_IMAGES}img/{COARSE_NEB_FMAX}fmax/{COARSE_NEB_STEPS}steps")
print(f"Fine NEB:   {FINE_N_IMAGES}img/{FINE_NEB_FMAX}fmax/{FINE_NEB_STEPS}steps")
print(f"KSP: {K_SHORTEST} paths")

## 3. MP 数据获取与中间相过滤

In [ ]:
adaptor = AseAtomsAdaptor()
t0 = time()

with MPRester(API_KEY) as mpr:
    docs = mpr.materials.summary.search(formula=TARGET_FORMULA,
        fields=["material_id","structure","energy_above_hull"])
    docs = sorted([d for d in docs if d.energy_above_hull is not None],
                  key=lambda x: x.energy_above_hull)
    target_struct = docs[0].structure
    target_id = docs[0].material_id
    print(f"Target: {target_struct.composition.reduced_formula} ({target_id})")
    print(f"  e_above_hull = {docs[0].energy_above_hull:.4f} eV/atom")

    els = [str(el) for el in target_struct.composition.elements]
    entries = mpr.get_entries_in_chemsys(els, include_structure=True,
        additional_criteria={"thermo_types": ["GGA_GGA+U"]})
    scheme = MaterialsProjectDFTMixingScheme()
    entries = scheme.process_entries(entries)
    print(f"Chemical system: {els}, {len(entries)} MP entries")

    all_formulas = sorted(set(e.composition.reduced_formula for e in entries))
    print(f"Pre-fetching structures for {len(all_formulas)} unique phases...")
    structure_cache = {}
    for formula in all_formulas:
        docs_f = mpr.materials.summary.search(formula=formula,
            fields=["material_id","structure","energy_above_hull"])
        docs_f = sorted([d for d in docs_f if d.energy_above_hull is not None],
                        key=lambda x: x.energy_above_hull)
        if docs_f:
            structure_cache[formula] = docs_f[0].structure
    print(f"  Cached {len(structure_cache)} structures")

pd_0k = PhaseDiagram(entries)
filtered = []
for e in entries:
    hull_e = pd_0k.get_e_above_hull(e)
    if hull_e is not None and hull_e < E_ABOVE_HULL_CUTOFF:
        e.data["e_above_hull"] = hull_e
        filtered.append(e)

formulas = sorted(set(e.composition.reduced_formula for e in filtered))
print(f"\nAfter hull filter (< {E_ABOVE_HULL_CUTOFF} eV/atom): {len(filtered)} entries, {len(formulas)} phases")
print(f"Phases: {formulas}")

# Keep only ground-state polymorph per formula (prevents combinatorial explosion)
ground_state = {}
for e in filtered:
    rf = e.composition.reduced_formula
    if rf not in ground_state or e.energy_per_atom < ground_state[rf].energy_per_atom:
        ground_state[rf] = e
filtered = list(ground_state.values())
formulas = sorted(ground_state.keys())
print(f"After polymorph dedup: {len(filtered)} ground-state entries, {len(formulas)} phases")

target_comp = target_struct.composition
decomp = pd_0k.get_decomposition(target_comp)
if len(decomp) <= 1:
    target_entries_local = [e for e in entries if e.composition.reduced_formula == target_comp.reduced_formula]
    entries_no_target = [e for e in entries if e not in target_entries_local]
    decomp = PhaseDiagram(entries_no_target).get_decomposition(target_comp)

if USE_MANUAL_PRECURSORS:
    PRECURSOR_FORMULAS = MANUAL_PRECURSORS
    print(f"Precursors (manual): {PRECURSOR_FORMULAS}")
else:
    PRECURSOR_FORMULAS = [pd_e.composition.reduced_formula for pd_e in decomp.keys()]
    print(f"Precursors (auto): {PRECURSOR_FORMULAS}")

# ---- Composition window (optional) ----
if COMPOSITION_WINDOW:
    ref_formulas = PRECURSOR_FORMULAS + [TARGET_FORMULA]
    elem_range = {}
    for rf in ref_formulas:
        comp = target_struct.composition if rf == TARGET_FORMULA else None
        if comp is None:
            for e in entries:
                if e.composition.reduced_formula == rf:
                    comp = e.composition; break
        if comp is None: continue
        for el, amt in comp.element_composition.items():
            frac = amt / comp.num_atoms
            lo, hi = elem_range.get(el, [frac, frac])
            elem_range[el] = [min(lo, frac), max(hi, frac)]
    windowed = []
    for e in filtered:
        ok = True
        for el, amt in e.composition.element_composition.items():
            frac = amt / e.composition.num_atoms
            lo, hi = elem_range.get(el, [0.0, 1.0])
            if frac < lo - 0.05 or frac > hi + 0.05:
                ok = False; break
        if ok: windowed.append(e)
    n_removed = len(filtered) - len(windowed)
    filtered = windowed
    formulas = sorted(set(e.composition.reduced_formula for e in filtered))
    if n_removed > 0:
        print(f"Composition window removed {n_removed} phases (outside precursor-target range)")
        print(f"After window filter: {len(filtered)} entries, {len(formulas)} phases")

print(f"\nSetup done in {time()-t0:.0f}s")

## 4. 枚举反应与去重

In [ ]:
def enumerate_reactions(entries, max_reactants=2):
    entry_list = list(entries)
    combos = []
    for n in range(1, max_reactants + 1):
        for combo in combinations(entry_list, n):
            combos.append(set(combo))
    print(f"Phase combinations: {len(combos)}")
    reactions = []
    for i in range(len(combos)):
        for j in range(i + 1, len(combos)):
            if combos[i] & combos[j]:
                continue
            try:
                rxn = ComputedReaction.balance(list(combos[i]), list(combos[j]))
                if not rxn.is_identity:
                    reactions.append(rxn)
                    reactions.append(rxn.reverse())
            except Exception:
                pass
    print(f"Raw reactions: {len(reactions)}")
    return reactions

def deduplicate_by_formula(reactions):
    seen = {}
    unique = []
    for rxn in reactions:
        r_key = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        p_key = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        key = (r_key, p_key)
        if key not in seen:
            seen[key] = rxn
            unique.append(rxn)
    return unique

rxns_raw = enumerate_reactions(filtered, MAX_REACTANTS)
rxns_unique = deduplicate_by_formula(rxns_raw)
print(f"After dedup: {len(rxns_unique)} unique reaction edges")

# ---- Only keep reactions relevant to synthesis ----
precursor_set = frozenset(PRECURSOR_FORMULAS)
rxns_filtered = []
for rxn in rxns_unique:
    r_fs = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
    p_fs = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
    if (r_fs & precursor_set) or (TARGET_FORMULA in p_fs) or (TARGET_FORMULA in r_fs):
        rxns_filtered.append(rxn)
rxns_unique = rxns_filtered
print(f"After precursor/target proximity filter: {len(rxns_unique)} edges")

n_edges = len(rxns_unique)
print(f"\nEstimated CI-NEB time:")
print(f"  Coarse: ~{n_edges*40/60:.0f}-{n_edges*60/60:.0f} min ({n_edges} edges)")
print(f"  Fine:   ~5-15 min (top-3 path edges only)")

## 5. MACE 力场初始化

In [ ]:
# GPU detection
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            MACE_DEVICE = "cuda"
            print(f"GPU detected: {torch.cuda.get_device_name(0)}")
        else:
            print("GPU not available, using CPU")
    except ImportError:
        print("torch not found, using CPU")
print(f"Loading MACE-MP-0 ({MACE_MODEL}, {MACE_DEVICE}, {MACE_DTYPE}) ...", flush=True)
mace_calc = mace_mp(model=MACE_MODEL, device=MACE_DEVICE, default_dtype=MACE_DTYPE)
print("MACE ready")

## 6. CI-NEB 工具函数


In [ ]:
def hungarian_match_atoms(atoms_A, atoms_B):
    """匈牙利算法：按元素分组最小化PBC感知距离"""
    if sorted(atoms_A.get_chemical_symbols()) != sorted(atoms_B.get_chemical_symbols()):
        raise ValueError("Chemical symbols mismatch!")
    n = len(atoms_A)
    elements = sorted(set(atoms_A.get_chemical_symbols()))
    new_order = np.zeros(n, dtype=int)
    for el in elements:
        idx_A = [i for i, s in enumerate(atoms_A.get_chemical_symbols()) if s == el]
        idx_B = [i for i, s in enumerate(atoms_B.get_chemical_symbols()) if s == el]
        if not idx_A: continue
        pos_A = atoms_A.positions[idx_A]; pos_B = atoms_B.positions[idx_B]
        dist_matrix = np.zeros((len(idx_A), len(idx_B)))
        for a in range(len(idx_A)):
            for b in range(len(idx_B)):
                delta = pos_A[a] - pos_B[b]
                if atoms_A.pbc.any():
                    cell_inv = np.linalg.inv(atoms_A.get_cell().T).T
                    delta_frac = np.dot(delta, cell_inv)
                    delta_frac -= np.round(delta_frac)
                    delta = np.dot(delta_frac, atoms_A.get_cell())
                dist_matrix[a, b] = np.linalg.norm(delta)
        row_ind, col_ind = linear_sum_assignment(dist_matrix)
        for a, b in zip(row_ind, col_ind):
            new_order[idx_A[a]] = idx_B[b]
    return atoms_B.copy()[new_order]

def build_neb_endpoints(rxn, structure_cache, adaptor):
    """用反应配平系数构建化学计量正确的CI-NEB端点
    
    策略：将每相 primitive 视为"原子块"，枚举整数乘子直到元素平衡。
    """
    from fractions import Fraction
    from collections import Counter
    from itertools import product as _product
    coeffs = rxn.coefficients
    r_entries = list(rxn.reactant_entries)
    p_entries = list(rxn.product_entries)
    
    # 整数系数
    denoms = [Fraction(abs(c)).limit_denominator(100).denominator for c in coeffs]
    lcm_d = 1
    for d in denoms: lcm_d = lcm_d * d // gcd(lcm_d, d)
    
    def get_prim(entry):
        s = structure_cache.get(entry.composition.reduced_formula)
        return adaptor.get_atoms(s) if s else None
    
    r_prims = [get_prim(e) for e in r_entries]
    p_prims = [get_prim(e) for e in p_entries]
    if any(a is None for a in r_prims + p_prims):
        return None, None
    
    def cell_elements(atoms):
        return Counter(atoms.get_chemical_symbols())
    
    r_els = [cell_elements(a) for a in r_prims]  # per-primitive-cell element count
    p_els = [cell_elements(a) for a in p_prims]
    
    # Brute-force search for integer multipliers (max ~50 cells per phase)
    MAX_M = 50
    best = None
    best_score = 10**9
    
    for r_ms in _product(*[range(1, MAX_M+1) for __ in r_entries]):
        r_el = Counter()
        for cel, m in zip(r_els, r_ms):
            for el, n in cel.items():
                r_el[el] += n * m
        r_total = sum(r_el.values())
        
        for p_ms in _product(*[range(1, MAX_M+1) for __ in p_entries]):
            p_el = Counter()
            for cel, m in zip(p_els, p_ms):
                for el, n in cel.items():
                    p_el[el] += n * m
            p_total = sum(p_el.values())
            
            # 判定：元素数目完全相等 + 总原子数不过大
            if r_el == p_el and r_total > 0:
                score = r_total  # 越小越好
                if score < best_score:
                    best_score = score
                    best = (list(r_ms), list(p_ms))
                    # 找到最小的就提前退出当前内循环（不必穷举）
                    break
            if p_total >= best_score:
                break  # 乘积侧已经太大，跳过
        if r_total >= best_score:
            break  # 反应物侧已经太大
        if best is not None and best_score <= 100:
            break  # 已找到很小的解
    
    if best is None:
        return None, None
    
    r_ms, p_ms = best
    
    # 构建原子块
    def build_side(prims, mults):
        atoms_list = []
        for a, m in zip(prims, mults):
            aa = a.copy()
            if m > 1:
                aa = a * [m, 1, 1]
            atoms_list.append(aa)
        return atoms_list
    
    r_atoms_raw = build_side(r_prims, r_ms)
    p_atoms_raw = build_side(p_prims, p_ms)
    
    # 合并在一个盒子
    box = 25.0
    initial_atoms = r_atoms_raw[0].copy()
    shift = 6.0
    for a in r_atoms_raw[1:]:
        ac = a.copy(); ac.translate([shift, 0, 0])
        initial_atoms += ac; shift += 6.0
    initial_atoms.set_cell([box, box, box]); initial_atoms.center(); initial_atoms.set_pbc(True)
    
    final_atoms = p_atoms_raw[0].copy()
    shift = 6.0
    for a in p_atoms_raw[1:]:
        ac = a.copy(); ac.translate([shift, 0, 0])
        final_atoms += ac; shift += 6.0
    final_atoms.set_cell([box, box, box]); final_atoms.center(); final_atoms.set_pbc(True)
    
    # 排序 + 匈牙利匹配
    i_idx = np.argsort(initial_atoms.get_atomic_numbers(), kind="mergesort")
    f_idx = np.argsort(final_atoms.get_atomic_numbers(), kind="mergesort")
    initial_atoms = initial_atoms[i_idx]; final_atoms = final_atoms[f_idx]
    if initial_atoms.get_chemical_symbols() != final_atoms.get_chemical_symbols():
        try:
            final_atoms = hungarian_match_atoms(initial_atoms, final_atoms)
        except ValueError:
            return None, None
    return initial_atoms, final_atoms

def run_ci_neb_edge(rxn, structure_cache, adaptor, mace_calc,
                     n_images=7, fmax=0.5, max_steps=500, label="", verbose=True):
    """对一条反应边计算 CI-NEB 能垒（复用 mace_calc）"""
    initial_atoms, final_atoms = build_neb_endpoints(
        rxn, structure_cache, adaptor)
    if initial_atoms is None:
        if verbose: print(f"  SKIP: cannot build endpoints")
        return None, None
    images = [initial_atoms.copy()]
    images += [initial_atoms.copy() for _ in range(n_images - 2)]
    images += [final_atoms.copy()]
    try:
        idpp_interpolate(images, fmax=fmax)
    except Exception:
        for i in range(1, n_images - 1):
            frac = i / (n_images - 1)
            images[i].set_positions(
                initial_atoms.positions * (1 - frac) + final_atoms.positions * frac)
    for a in images:
        a.calc = mace_calc
    neb = NEB(images, climb=True, k=1.0, method="improvedtangent", allow_shared_calculator=True)
    FIRE(neb).run(fmax=fmax, steps=max_steps)
    energies = np.array([a.get_potential_energy() for a in images])
    energies -= energies[0]
    fwd = max(energies.max(), 0.0)
    rev = max(energies.max() - energies[-1], 0.0)
    if verbose:
        print(f"  {label}: fwd={fwd:.3f} eV, rev={rev:.3f} eV, dE={energies[-1]:.3f}")
    return fwd, rev

print("CI-NEB toolkit ready")

## 7. 全边粗筛 CI-NEB


In [ ]:
coarse_barriers = {}
t_start = time()
n_total = len(rxns_unique)

for idx, rxn in enumerate(rxns_unique):
    r_key = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
    p_key = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
    if (r_key, p_key) in coarse_barriers:
        continue
    r_str = "+".join(sorted(r_key))
    p_str = "+".join(sorted(p_key))
    label = f"[{idx+1}/{n_total}] {r_str}->{p_str}"
    fwd, rev = run_ci_neb_edge(
        rxn, structure_cache, adaptor, mace_calc,
        n_images=COARSE_N_IMAGES, fmax=COARSE_NEB_FMAX, max_steps=COARSE_NEB_STEPS,
        label=label, verbose=True)
    if fwd is not None:
        coarse_barriers[(r_key, p_key)] = (fwd, rev)
    elapsed = time() - t_start
    if DEBUG_MODE and (idx + 1) >= DEBUG_COARSE_EDGES:
        print(f"  Debug mode: stopping after {DEBUG_COARSE_EDGES} edges")
        break
    if (idx + 1) % 10 == 0 or idx == 0:
        avg = elapsed / (idx + 1)
        eta = avg * (n_total - idx - 1)
        print(f"  --- {idx+1}/{n_total} done ({elapsed:.0f}s, ETA {eta:.0f}s) ---")

# Force forward/reverse barrier consistency (coarse NEB may be asymmetric)
for (rk, pk), (fwd, rev) in coarse_barriers.items():
    rev_key = (pk, rk)
    if rev_key in coarse_barriers:
        f2, r2 = coarse_barriers[rev_key]
        coarse_barriers[(rk, pk)] = (max(fwd, r2), max(rev, f2))

print(f"\nCoarse CI-NEB done: {len(coarse_barriers)}/{n_total} edges in {time()-t_start:.0f}s")

## 8. 粗筛能垒建图与 KSP 搜索

In [ ]:
def build_kinetic_graph(rxns, barrier_dict):
    """用CI-NEB能垒作为边成本构建有向反应图"""
    G = nx.DiGraph()
    node_formulas = {}
    unknown_count = 0
    for rxn in rxns:
        r_key = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        p_key = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        def get_node(key):
            for n, v in node_formulas.items():
                if n >= 0 and v == key:
                    return n
            new_idx = len(node_formulas)
            G.add_node(new_idx, label=" + ".join(sorted(key)))
            node_formulas[new_idx] = key
            return new_idx
        r_idx = get_node(r_key)
        p_idx = get_node(p_key)
        bi = barrier_dict.get((r_key, p_key))
        cost = bi[0] if bi is not None else 10.0
        if bi is None: unknown_count += 1
        G.add_edge(r_idx, p_idx, reaction=rxn, cost=cost, r_formulas=r_key, p_formulas=p_key)
    lb = 0
    for p_idx in list(G.nodes()):
        if p_idx < 0: continue
        pk = node_formulas.get(p_idx)
        for r_idx in list(G.nodes()):
            if r_idx < 0: continue
            if node_formulas.get(r_idx) == pk:
                G.add_edge(p_idx, r_idx, reaction="loopback", cost=0.0)
                lb += 1
    if unknown_count:
        print(f"  Warning: {unknown_count} edges without barrier (penalty cost=10)")
    print(f"  Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges ({lb} loopback)")
    return G, node_formulas

def add_source_sink(G, node_formulas, source_formulas, target_formula):
    SOURCE, SINK = -1, -2
    G.add_node(SOURCE, label="SOURCE")
    G.add_node(SINK, label="SINK")
    source_set = frozenset(source_formulas)
    src = snk = 0
    for n, n_key in node_formulas.items():
        if n < 0: continue
        if n_key.issubset(source_set):
            G.add_edge(SOURCE, n, reaction="precursor_edge", cost=0.0); src += 1
        if target_formula in n_key:
            G.add_edge(n, SINK, reaction="target_edge", cost=0.0); snk += 1
    print(f"  Source: {src}, Sink: {snk}")
    return SOURCE, SINK

def extract_pathway(G, node_path):
    rxns, costs = [], []
    for i in range(len(node_path) - 1):
        edge = G[node_path[i]][node_path[i+1]]
        rxn = edge.get("reaction")
        if rxn is not None and not isinstance(rxn, str):
            rxns.append(rxn)
        costs.append(edge.get("cost", 0))
    return rxns, costs

def yen_ksp(G, source, target, K=10):
    """Yen's K-Shortest Paths"""
    try:
        fp = nx.shortest_path(G, source, target, weight="cost")
        fc = sum(G[fp[i]][fp[i+1]]["cost"] for i in range(len(fp)-1))
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return []
    A = [(fc, fp)]
    B = []
    for k in range(1, K):
        prev = A[-1][1]
        for i in range(len(prev) - 1):
            spur = prev[i]
            root = prev[:i+1]
            Gc = G.copy()
            for _, pv in A:
                if len(pv) > i + 1 and pv[:i+1] == root:
                    u, v = pv[i], pv[i+1]
                    if Gc.has_edge(u, v): Gc.remove_edge(u, v)
            for node in root[:-1]:
                if node != spur and node >= 0:
                    Gc.remove_node(node)
            try:
                sp = nx.shortest_path(Gc, spur, target, weight="cost")
                tp = root[:-1] + sp
                tc = sum(G[tp[j]][tp[j+1]]["cost"] for j in range(len(tp)-1)
                        if G.has_edge(tp[j], tp[j+1]))
                if tc > 0: B.append((tc, tp))
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                pass
        if not B: break
        B.sort(key=lambda x: x[0])
        found = False; new_B = []
        for cv, pv in B:
            if pv not in [p for _, p in A]:
                A.append((cv, pv)); found = True
            else: new_B.append((cv, pv))
        B = new_B
        if not found: break
    return A

G, node_formulas = build_kinetic_graph(rxns_unique, coarse_barriers)
SOURCE, SINK = add_source_sink(G, node_formulas, PRECURSOR_FORMULAS, TARGET_FORMULA)
print(f"Total: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

ksp_coarse = yen_ksp(G, SOURCE, SINK, K=K_SHORTEST)
print(f"\nCoarse-barrier KSP: {len(ksp_coarse)} paths\n")
for rank, (tc, path) in enumerate(ksp_coarse[:5]):
    rxns_p, _ = extract_pathway(G, path)
    print(f"--- Path {rank+1} (barrier={tc:.2f} eV, {len(rxns_p)} steps) ---")
    for j, rxn in enumerate(rxns_p):
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        fwd, _ = coarse_barriers.get((rk, pk), (None, None))
        bs = f"{fwd:.2f}eV" if fwd else "N/A"
        print(f"  {j+1}. [{bs}] {rxn}")
    print()

## 9. Top-3 路径边精修 CI-NEB


In [ ]:
if DEBUG_MODE:
    FINE_N_IMAGES = 5    # 调试模式降级到粗筛
    FINE_NEB_FMAX = 1.0
    FINE_NEB_STEPS = 200

fine_edge_keys = set()
for _, path in ksp_coarse[:3]:
    for i in range(len(path) - 1):
        edge = G[path[i]][path[i+1]]
        if edge.get("reaction") is not None and not isinstance(edge.get("reaction"), str):
            rk = edge.get("r_formulas")
            pk = edge.get("p_formulas")
            if rk and pk: fine_edge_keys.add((rk, pk))

print(f"Unique edges in top-3 paths: {len(fine_edge_keys)}")
for rk, pk in fine_edge_keys:
    rs = "+".join(sorted(rk)); ps = "+".join(sorted(pk))
    cf = coarse_barriers.get((rk, pk), (None, None))[0]
    print(f"  {rs} -> {ps}  (coarse={cf:.2f}eV)" if cf else f"  {rs} -> {ps}")

print(f"\nRunning fine CI-NEB ({FINE_N_IMAGES}img/{FINE_NEB_FMAX}fmax)...")
t_fine = time()
fine_barriers = {}
for idx, (rk, pk) in enumerate(fine_edge_keys):
    rs = "+".join(sorted(rk)); ps = "+".join(sorted(pk))
    label = f"Fine [{idx+1}/{len(fine_edge_keys)}] {rs}->{ps}"
    matching_rxn = None
    for r in rxns_unique:
        r_rk = frozenset(e.composition.reduced_formula for e in r.reactant_entries)
        r_pk = frozenset(e.composition.reduced_formula for e in r.product_entries)
        if r_rk == rk and r_pk == pk:
            matching_rxn = r; break
    if matching_rxn is None: continue
    fwd, rev = run_ci_neb_edge(
        matching_rxn, structure_cache, adaptor, mace_calc,
        n_images=FINE_N_IMAGES, fmax=FINE_NEB_FMAX, max_steps=FINE_NEB_STEPS,
        label=label, verbose=True)
    if fwd is not None: fine_barriers[(rk, pk)] = (fwd, rev)
    if DEBUG_MODE and len(fine_barriers) >= DEBUG_FINE_EDGES:
        print(f"  Debug mode: stopping after {DEBUG_FINE_EDGES} fine edges")
        break
print(f"Fine CI-NEB done in {time()-t_fine:.0f}s: {len(fine_barriers)}/{len(fine_edge_keys)} edges")

## 10. 精修能垒重建图与最终合成路径

In [ ]:
final_barriers = dict(coarse_barriers)
final_barriers.update(fine_barriers)

G_final, node_formulas_final = build_kinetic_graph(rxns_unique, final_barriers)
SOURCE_F, SINK_F = add_source_sink(G_final, node_formulas_final, PRECURSOR_FORMULAS, TARGET_FORMULA)

ksp_final = yen_ksp(G_final, SOURCE_F, SINK_F, K=K_SHORTEST)
print(f"\nFinal synthesis KSP: {len(ksp_final)} paths\n")
for rank, (tc, path) in enumerate(ksp_final[:5]):
    rxns_p, _ = extract_pathway(G_final, path)
    tags = []
    for rxn in rxns_p:
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        tags.append("*" if (rk, pk) in fine_barriers else " ")
    print(f"--- Synthesis Path {rank+1} (barrier={tc:.2f} eV, {len(rxns_p)} steps) ---")
    for j, rxn in enumerate(rxns_p):
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        fwd, rev = final_barriers.get((rk, pk), (None, None))
        print(f"  {j+1}. [{fwd:.2f}/{rev:.2f}eV]{tags[j]} {rxn}" if fwd else f"  {j+1}. {rxn}")
    print()

## 11. 分解反应路径

In [ ]:
G_dec = G_final.copy()
for old in [-1, -2]:
    if old in G_dec.nodes(): G_dec.remove_node(old)
SOURCE_DEC, SINK_DEC = -1, -2
G_dec.add_node(SOURCE_DEC, label="SOURCE_DEC")
G_dec.add_node(SINK_DEC, label="SINK_DEC")
precursor_set = frozenset(PRECURSOR_FORMULAS)
src_dec = snk_dec = 0
for n, n_key in node_formulas_final.items():
    if n < 0: continue
    if TARGET_FORMULA in n_key:
        G_dec.add_edge(SOURCE_DEC, n, reaction="precursor_edge", cost=0.0); src_dec += 1
    if n_key.issubset(precursor_set):
        G_dec.add_edge(n, SINK_DEC, reaction="target_edge", cost=0.0); snk_dec += 1
print(f"Decomposition graph: source={src_dec}, sink={snk_dec}")

ksp_dec = yen_ksp(G_dec, SOURCE_DEC, SINK_DEC, K=K_SHORTEST)
print(f"\nDecomposition KSP: {len(ksp_dec)} paths\n")
for rank, (tc, path) in enumerate(ksp_dec[:5]):
    rxns_p, _ = extract_pathway(G_dec, path)
    print(f"--- Decomp Path {rank+1} (barrier={tc:.2f} eV, {len(rxns_p)} steps) ---")
    for j, rxn in enumerate(rxns_p):
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        fwd, rev = final_barriers.get((rk, pk), (None, None))
        print(f"  {j+1}. [{fwd:.2f}/{rev:.2f}eV] {rxn}" if fwd else f"  {j+1}. {rxn}")
    print()

## 12. 可视化：合成与分解路径能垒

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Synthesis ---
ax = axes[0]
for rank, (cost, path) in enumerate(ksp_final[:3]):
    # Extract node labels (skip SOURCE/SINK virtual nodes)
    labels = []
    for n in path:
        if n >= 0:  # real nodes only
            labels.append(G_final.nodes[n].get("label", str(n)))
    rxns_p, _ = extract_pathway(G_final, path)
    barriers = []
    for rxn in rxns_p:
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        fwd, _ = final_barriers.get((rk, pk), (0, 0)); barriers.append(fwd or 0)
    cum = np.cumsum([0] + barriers)
    ax.plot(range(len(cum)), cum, "-o", lw=2, ms=8, label=f"Path {rank+1} (sum={cost:.1f} eV)")
ax.axhline(0, color="gray", ls="--")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=7, rotation=30, ha="right")
ax.set_ylabel("Cumulative CI-NEB barrier (eV)")
ax.set_title(f"Synthesis: {"+".join(PRECURSOR_FORMULAS)} -> {TARGET_FORMULA}")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Decomposition ---
ax = axes[1]
for rank, (cost, path) in enumerate(ksp_dec[:3]):
    labels = []
    for n in path:
        if n >= 0:
            labels.append(G_dec.nodes[n].get("label", str(n)))
    rxns_p, _ = extract_pathway(G_dec, path)
    barriers = []
    for rxn in rxns_p:
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        fwd, _ = final_barriers.get((rk, pk), (0, 0)); barriers.append(fwd or 0)
    cum = np.cumsum([0] + barriers)
    ax.plot(range(len(cum)), cum, "-s", lw=2, ms=8, label=f"Path {rank+1} (sum={cost:.1f} eV)")
ax.axhline(0, color="gray", ls="--")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=7, rotation=30, ha="right")
ax.set_ylabel("Cumulative CI-NEB barrier (eV)")
ax.set_title(f"Decomposition: {TARGET_FORMULA} -> {"+".join(PRECURSOR_FORMULAS)}")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 13. 汇总输出与报告保存

In [ ]:
print("=" * 70)
print(f"  Pure Kinetic Reaction Network + CI-NEB Evaluation")
print(f"  Target: {TARGET_FORMULA} ({target_id})")
print(f"  Precursors: {PRECURSOR_FORMULAS}")
print(f"  Chemical system: {els}")
print("=" * 70)

print(f"\n  Phase filtering: e_above_hull < {E_ABOVE_HULL_CUTOFF} eV/atom -> {len(formulas)} phases")
print(f"  Reaction edges: {len(rxns_unique)} unique")
print(f"  Coarse CI-NEB: {len(coarse_barriers)} edges")
print(f"  Fine CI-NEB:   {len(fine_barriers)} edges")
if len(ksp_final) > 0:
    best_syn = ksp_final[0]
    rxns_best, _ = extract_pathway(G_final, best_syn[1])
    print(f"\n  === Best synthesis pathway (barrier={best_syn[0]:.2f} eV) ===")
    for j, rxn in enumerate(rxns_best):
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        fwd, rev = final_barriers.get((rk, pk), (None, None))
        print(f"    {j+1}. fwd={fwd:.2f}eV rev={rev:.2f}eV | {rxn}" if fwd else f"    {j+1}. {rxn}")
if len(ksp_dec) > 0:
    best_dec = ksp_dec[0]
    rxns_dec, _ = extract_pathway(G_dec, best_dec[1])
    print(f"\n  === Best decomposition pathway (barrier={best_dec[0]:.2f} eV) ===")
    for j, rxn in enumerate(rxns_dec):
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        fwd, rev = final_barriers.get((rk, pk), (None, None))
        print(f"    {j+1}. fwd={fwd:.2f}eV rev={rev:.2f}eV | {rxn}" if fwd else f"    {j+1}. {rxn}")
with open("kinetics_network_report.md", "w", encoding="utf-8") as f:
    pfx = " + ".join(PRECURSOR_FORMULAS)
    f.write(f"# {TARGET_FORMULA} Pure Kinetic Reaction Network\n\n")
    f.write(f"**Target**: {TARGET_FORMULA} ({target_id})\n")
    f.write(f"**Precursors**: {pfx}\n")
    f.write(f"**MACE**: MACE-MP-0 ({MACE_MODEL})\n")
    f.write(f"**Method**: Coarse->Fine CI-NEB weighted reaction network\n\n")
    f.write("## Synthesis\n\n")
    for rank, (cost, path) in enumerate(ksp_final[:5]):
        rxns_p, _ = extract_pathway(G_final, path)
        f.write(f"### Path {rank+1} (total barrier = {cost:.2f} eV)\n\n")
        for j, rxn in enumerate(rxns_p):
            rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
            pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
            fwd, rev = final_barriers.get((rk, pk), (None, None))
            s = f"{j+1}. fwd={fwd:.2f}eV rev={rev:.2f}eV | {rxn}\n" if fwd else f"{j+1}. {rxn}\n"
            f.write(s)
        f.write("\n")
    f.write("## Decomposition\n\n")
    for rank, (cost, path) in enumerate(ksp_dec[:3]):
        rxns_p, _ = extract_pathway(G_dec, path)
        f.write(f"### Path {rank+1} (total barrier = {cost:.2f} eV)\n\n")
        for j, rxn in enumerate(rxns_p):
            rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
            pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
            fwd, rev = final_barriers.get((rk, pk), (None, None))
            s = f"{j+1}. fwd={fwd:.2f}eV rev={rev:.2f}eV | {rxn}\n" if fwd else f"{j+1}. {rxn}\n"
            f.write(s)
        f.write("\n")
print("\nkinetics_network_report.md saved")